# АХ: Агент-художник — «Непереводимое»

Выполнили: *(впишите ФИО)*

**Идея.** В каждом языке есть слова, которые не переводятся одним словом: русская *тоска*, японское *komorebi*, валлийское *hiraeth*…
Мы проверяем, может ли многоагентная система перевести такое слово **на язык изображения** — так, чтобы зритель,
который этого слова не знает, понял его смысл.

Отсюда главная особенность работы — **«слепой» критик**: VLM смотрит на картину, *не зная, какое слово рисовали*,
и пытается сам сформулировать, какое чувство или понятие она выражает. Отдельный агент-судья сравнивает эту
догадку с настоящим значением слова и ставит оценку 0–10. Если зритель «не понял» картину, его интерпретация
и подсказки судьи возвращаются художнику и промпт-инженеру, и они спорят о новой версии.
Метрика качества здесь — не «красиво ли», а **«дошёл ли смысл»**, то есть по сути игра в «Крокодил» между нейросетями.

### Агенты

| Агент | Модель (Yandex AI Studio) | Роль |
|---|---|---|
| 🎨 Художник | Alice AI LLM | придумывает визуальную метафору, отстаивает идею |
| 🛠 Промпт-инженер | DeepSeek V4.1 Flash | знает ограничения генератора, спорит, упрощает, пишет финальный промпт |
| 🖼 Генератор | Alice AI ART 3.0 | рисует картину по промпту (Images API) |
| 👁 Слепой критик | Qwen3.6 35B (VLM) | видит только картинку, описывает её и угадывает смысл |
| ⚖️ Судья | Qwen3.6 35B (без картинки) | сравнивает догадку критика со значением слова, ставит балл |
| 🏛 Куратор | Alice AI LLM | выбирает лучшую версию и пишет этикетку для выставки |

### Схема одного произведения

```
             ┌──────────── обратная связь: что увидел критик + чего не хватило ────────────┐
             ▼                                                                             │
 слово + смысл ──► 🎨 Художник ⇄ 🛠 Промпт-инженер ──► PROMPT ──► 🖼 Alice AI ART ──► 👁 Слепой критик ──► ⚖️ Судья
                   (диалог из нескольких реплик)                                        (не знает слова)    балл 0..10
                                                                                                                │
                                              балл ≥ порога или исчерпаны итерации ──► 🏛 Куратор ──► этикетка + сайт
```

Художник и промпт-инженер работают на **разных LLM**, поэтому это действительно диалог двух моделей,
а не одна модель, разговаривающая сама с собой. Все реплики сохраняются и публикуются на сайте вместе с картинами.

## 1. Настройка

Ключи берутся из файла `.env` (см. `.env.sample`): `folder_id` — идентификатор каталога Yandex Cloud,
`api_key` — API-ключ сервисного аккаунта с ролью `ai.languageModels.user` / `ai.imageGeneration.user`.

In [ ]:
from dotenv import load_dotenv
import os

if load_dotenv(override=True):
    print(".env loaded")

folder_id = os.environ['folder_id']
api_key = os.environ['api_key']

In [ ]:
import base64, io, json, re, time, datetime
from pathlib import Path

import openai
from openai import OpenAI
from PIL import Image
from IPython.display import display, Markdown, HTML

# Все модели Yandex AI Studio доступны через OpenAI-совместимый endpoint:
# текст и VLM — через Responses API, картинки — через Images API.
client = OpenAI(
    base_url="https://ai.api.cloud.yandex.net/v1",
    api_key=api_key,
    project=folder_id,
    max_retries=5,      # клиент сам повторяет запросы при 429/5xx
    timeout=300,
)

MODELS = {
    "artist":   "aliceai-llm",
    "engineer": "deepseek-v4.1-flash",
    "critic":   "qwen3.6-35b-a3b",   # мультимодальная модель, принимает изображения в base64
    "judge":    "qwen3.6-35b-a3b",
    "curator":  "aliceai-llm",
}
IMAGE_MODEL = "aliceai-image-art-3.0"
IMAGE_PROMPT_LIMIT = 500   # ограничение Alice AI ART на длину промпта

MAX_ITERATIONS = 3     # сколько версий картины максимум делаем для одного слова
TARGET_SCORE = 8       # если судья поставил столько или больше — картина «дошла» до зрителя
DIALOGUE_ROUNDS = 2    # сколько обменов репликами художник ⇄ промпт-инженер перед каждой генерацией

OUT_DIR = Path("docs")              # всё сохраняем прямо в папку сайта GitHub Pages
IMG_DIR = OUT_DIR / "images"
DATA_DIR = OUT_DIR / "data"

def gpt(model):
    return f"gpt://{folder_id}/{model}"

## 2. Примитивы: агент с памятью

Каждый агент — это пара *(модель, системная инструкция)* плюс **собственная история диалога**.
Историю мы храним на клиенте и целиком передаём в `input` Responses API — так каждый агент помнит,
о чём уже договорились, а разные агенты остаются независимыми (у каждого своя «голова»).

Реплики, которыми агенты обмениваются, параллельно пишутся в общий журнал `LOG` — он потом уходит на сайт.

In [ ]:
LOG = []   # журнал текущего произведения: [{"speaker", "iteration", "text"}]

def clean(text):
    # Рассуждающие модели иногда возвращают блок <think>…</think> — зрителю он не нужен
    return re.sub(r"<think>.*?</think>", "", text or "", flags=re.S).strip()

def say(speaker, text, iteration):
    # Записать реплику в журнал и красиво показать её в ноутбуке
    LOG.append({"speaker": speaker, "iteration": iteration, "text": text})
    display(Markdown(f"**{speaker}** · *версия {iteration + 1}*\n\n> " + text.replace("\n", "\n> ")))


class Agent:
    def __init__(self, name, model, instructions, temperature=0.7):
        self.name, self.model, self.instructions, self.temperature = name, model, instructions, temperature
        self.history = []

    def reset(self):
        self.history = []

    def ask(self, text, image_b64=None, remember=True):
        # Отправить агенту сообщение (опционально с картинкой) и получить ответ
        if image_b64:
            content = [
                {"type": "input_text", "text": text},
                {"type": "input_image", "image_url": f"data:image/jpeg;base64,{image_b64}"},
            ]
        else:
            content = text
        messages = self.history + [{"role": "user", "content": content}]
        answer = clean(self._call(messages, text, image_b64))
        if remember:
            self.history += [{"role": "user", "content": content},
                             {"role": "assistant", "content": answer}]
        return answer

    def _call(self, messages, text, image_b64):
        try:
            res = client.responses.create(
                model=gpt(self.model),
                instructions=self.instructions,
                input=messages,
                temperature=self.temperature,
            )
            return res.output_text
        except openai.BadRequestError:
            if not image_b64:
                raise
            # Запасной путь для VLM: тот же запрос через Chat Completions (формат картинки чуть другой).
            # Критик всегда вызывается «с чистого листа», поэтому история здесь не нужна.
            res = client.chat.completions.create(
                model=gpt(self.model),
                temperature=self.temperature,
                messages=[
                    {"role": "system", "content": self.instructions},
                    {"role": "user", "content": [
                        {"type": "text", "text": text},
                        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"}},
                    ]},
                ],
            )
            return res.choices[0].message.content


def extract_json(text):
    # Достаём первый JSON-объект из ответа модели (модели любят оборачивать его в ```json … ```)
    m = re.search(r"\{.*\}", text or "", re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

def ask_json(agent, text, image_b64=None, attempts=3):
    # Спросить агента и получить разобранный JSON; при неудаче — переспросить
    for _ in range(attempts):
        data = extract_json(agent.ask(text, image_b64=image_b64, remember=False))
        if data is not None:
            return data
        text += "\n\nВНИМАНИЕ: ответь строго одним JSON-объектом, без пояснений вокруг."
    raise ValueError(f"{agent.name}: не удалось получить JSON")

## 3. Характеры агентов

Важные решения в промптах:

* **Художник** обязан искать *неочевидную* метафору и не использовать надписи — иначе критик просто прочитает слово с картины.
* **Промпт-инженер** знает слабые места генератора (текст, руки, многофигурные сцены, абстракции без предметов) и
  специально настроен **спорить**, а не соглашаться — так диалог становится содержательным.
* **Критик** ничего не знает о задумке, а **судья** — наоборот, не видит картину. Так оценка смысла отделена от оценки
  изображения, и ни один агент не может «подыграть» сам себе.

In [ ]:
ARTIST_PROMPT = """Ты — Художник-концептуалист. Вместе с коллегой, Промпт-инженером, ты создаёшь картину,
которая должна передать смысл непереводимого слова зрителю, который этого слова НЕ знает.
Правила:
- Ищи сильную, неочевидную визуальную метафору: сюжет, свет, цвет, композицию, стиль, материал.
- Никаких надписей, букв и символов-подсказок: смысл должен читаться из образа.
- Будь конкретен: что именно в кадре, где, при каком свете, в какой манере.
- Отстаивай идею, но честно принимай аргументы коллеги и замечания зрителей.
Отвечай по-русски, живо и коротко — не больше 120 слов."""

ENGINEER_PROMPT = f"""Ты — Промпт-инженер генеративной модели Alice AI ART. Ты работаешь в паре с Художником.
Ты хорошо знаешь возможности генератора:
- хорошо удаются: свет и атмосфера, фактуры, стилизация под живопись/фото/гравюру, один-два ясных объекта, пейзажи;
- плохо удаются: текст, руки и мелкие жесты, много персонажей, сложные взаимодействия, отвлечённые понятия без предметов,
  отрицания («без чего-то»).
Твоя задача — превратить идею Художника в то, что генератор действительно нарисует, и что зритель поймёт.
Задавай неудобные вопросы, спорь, предлагай упрощения и усиления. Не соглашайся просто так.
Отвечай по-русски, не больше 120 слов.
Когда тебя попросят написать финальный промпт, ответь РОВНО в формате:
PROMPT: <одно описание картины на русском языке, не длиннее {IMAGE_PROMPT_LIMIT - 50} символов: сюжет, композиция, свет, палитра, стиль>"""

CRITIC_PROMPT = """Ты — внимательный зритель-искусствовед на выставке. Перед тобой картина без подписи и без названия.
Ты ничего не знаешь о замысле автора. Посмотри на картину и ответь строго JSON-объектом:
{
  "description": "что изображено, 2-3 предложения",
  "feeling": "какое чувство, состояние или понятие, по-твоему, выражает картина — 1-2 предложения своими словами",
  "strengths": "что работает в картине, 1 предложение",
  "weaknesses": "что мешает понять картину или технические дефекты, 1 предложение"
}
Пиши по-русски."""

JUDGE_PROMPT = """Ты — беспристрастный судья в игре «Крокодил» между нейросетями. Художник рисовал непереводимое слово.
Зритель, не знавший слова, посмотрел на картину и сформулировал, что она выражает.
Оцени, насколько интерпретация зрителя совпадает со смыслом слова, по шкале от 0 до 10:
10 — зритель по сути назвал понятие со всеми ключевыми оттенками;
7 — уловил главное, но потерял важный оттенок;
4 — верная эмоциональная окраска, но смысл другой;
0 — интерпретация не связана со словом.
Ответь строго JSON-объектом:
{"score": <целое 0-10>, "matched": "что зритель уловил верно", "missing": "какой оттенок смысла потерян",
 "hint": "конкретный совет художнику: что изменить в образе, чтобы зритель понял точнее (не называя слово)"}
Пиши по-русски."""

CURATOR_PROMPT = """Ты — куратор выставки «Непереводимое», на которой нейросети пытаются нарисовать слова,
не имеющие перевода. Ты пишешь музейные этикетки: ёмко, точно, без пафоса и штампов, с одной неожиданной мыслью.
Пиши по-русски."""

artist   = Agent("🎨 Художник",       MODELS["artist"],   ARTIST_PROMPT,   temperature=0.9)
engineer = Agent("🛠 Промпт-инженер", MODELS["engineer"], ENGINEER_PROMPT, temperature=0.6)
critic   = Agent("👁 Слепой критик",  MODELS["critic"],   CRITIC_PROMPT,   temperature=0.3)
judge    = Agent("⚖️ Судья",          MODELS["judge"],    JUDGE_PROMPT,    temperature=0.1)
curator  = Agent("🏛 Куратор",        MODELS["curator"],  CURATOR_PROMPT,  temperature=0.8)

## 4. Генерация изображения и «взгляд» критика

Alice AI ART вызывается через OpenAI-совместимый Images API (`client.images.generate`) и возвращает картинку в base64.
Сохраняем её в JPEG — так сайт весит меньше. Для критика картинка дополнительно уменьшается до 768 px:
этого достаточно для анализа и заметно дешевле по токенам.

In [ ]:
def generate_image(prompt, path):
    res = client.images.generate(
        model=f"art://{folder_id}/{IMAGE_MODEL}",
        prompt=prompt,
        size="1024x1024",
    )
    img = Image.open(io.BytesIO(base64.b64decode(res.data[0].b64_json))).convert("RGB")
    path.parent.mkdir(parents=True, exist_ok=True)
    img.save(path, "JPEG", quality=90)
    return path

def image_b64(path, max_side=768):
    img = Image.open(path).convert("RGB")
    img.thumbnail((max_side, max_side))
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode()

## 5. Один цикл: обсуждение → промпт → картина → слепой критик → судья

* `discuss` — диалог Художника и Промпт-инженера. На первой итерации они получают слово и его смысл; на следующих —
  ещё и то, **что увидел критик** и **чего, по мнению судьи, не хватило**. Диалог заканчивается финальным `PROMPT:`.
* `render` — генерация; если генератор отклонил промпт (модерация, длина), инженер переписывает его.
* `evaluate` — критик смотрит на картину *с чистого листа* (без памяти о прошлых версиях, чтобы не было «привыкания»),
  судья сравнивает его интерпретацию со значением слова.

In [ ]:
def extract_prompt(text):
    m = re.search(r"PROMPT\s*:\s*(.+)", text, re.S | re.I)
    prompt = (m.group(1) if m else text).strip()
    return prompt.strip('"«»` \n')


def feedback_text(prev):
    # Обратная связь по предыдущей версии — общая для художника и инженера
    c, j = prev["critic"], prev["judge"]
    return (f"Прошлая версия (промпт: «{prev['prompt']}») получила {j['score']}/10.\n"
            f"Зритель, не знающий слова, увидел: {c['description']}\n"
            f"Он решил, что картина о том, что: {c['feeling']}\n"
            f"Судья: уловлено — {j['matched']}; потеряно — {j['missing']}.\n"
            f"Совет судьи: {j['hint']}")


def discuss(work, it, prev=None):
    if prev is None:
        task = (f"Новое слово: «{work['word']}» ({work['lang']}). Смысл: {work['meaning']}\n"
                "Предложи визуальную метафору для картины.")
    else:
        task = feedback_text(prev) + "\nПереосмысли образ так, чтобы зритель понял смысл точнее. Можно радикально."
    msg = artist.ask(task)
    say(artist.name, msg, it)

    for r in range(DIALOGUE_ROUNDS):
        context = (task + "\n\n") if r == 0 else ""
        reply = engineer.ask(f"{context}Художник: {msg}\n\nОбсуди идею: что генератор нарисует хорошо, что плохо, "
                             "что зритель может не понять. Задай вопрос или предложи изменение.")
        say(engineer.name, reply, it)
        msg = artist.ask(f"Промпт-инженер: {reply}\n\nОтветь ему"
                         + (" и зафиксируй окончательный образ." if r == DIALOGUE_ROUNDS - 1 else "."))
        say(artist.name, msg, it)

    prompt = extract_prompt(engineer.ask(f"Художник: {msg}\n\nНапиши финальный промпт для генератора."))
    for _ in range(3):
        if len(prompt) <= IMAGE_PROMPT_LIMIT:
            break
        prompt = extract_prompt(engineer.ask(
            f"Промпт длиной {len(prompt)} символов, а лимит {IMAGE_PROMPT_LIMIT}. Сократи, сохранив главное."))
    prompt = prompt[:IMAGE_PROMPT_LIMIT]
    say(engineer.name, "PROMPT: " + prompt, it)
    return prompt


def render(work, it, prompt):
    path = IMG_DIR / work["slug"] / f"v{it + 1}.jpg"
    for attempt in range(3):
        try:
            return prompt, generate_image(prompt, path)
        except openai.BadRequestError as e:
            # Например, сработала модерация — просим инженера переформулировать
            say("🖼 Генератор", f"Не могу нарисовать этот промпт: {e.message}", it)
            prompt = extract_prompt(engineer.ask(
                f"Генератор отклонил промпт с ошибкой: {e.message}\nПерепиши промпт, сохранив замысел."))[:IMAGE_PROMPT_LIMIT]
            say(engineer.name, "PROMPT: " + prompt, it)
    raise RuntimeError("Генератор трижды отклонил промпт")


def evaluate(work, it, path):
    seen = ask_json(critic, "Что ты видишь на этой картине и что она выражает?", image_b64=image_b64(path))
    seen = {k: str(seen.get(k, "")) for k in ("description", "feeling", "strengths", "weaknesses")}
    say(critic.name, f"{seen['description']}\n\n**Мне кажется, картина о том, что:** {seen['feeling']}\n\n"
                     f"*Сильное:* {seen['strengths']} *Слабое:* {seen['weaknesses']}", it)
    verdict = ask_json(judge,
        f"Слово: «{work['word']}» ({work['lang']}). Настоящий смысл: {work['meaning']}\n"
        f"Интерпретация зрителя: {seen.get('feeling', '')}\nОписание картины зрителем: {seen.get('description', '')}")
    try:
        score = max(0, min(10, int(round(float(verdict.get("score", 0))))))
    except (TypeError, ValueError):
        score = 0
    verdict = {"score": score, **{k: str(verdict.get(k, "")) for k in ("matched", "missing", "hint")}}
    say(judge.name, f"**{score}/10.** Уловлено: {verdict['matched']} "
                    f"Потеряно: {verdict['missing']} Совет: {verdict['hint']}", it)
    return seen, verdict

## 6. Полный цикл для одного слова и этикетка куратора

Повторяем цикл, пока судья не поставит `TARGET_SCORE` или не закончатся итерации. Лучшую версию
(с максимальным баллом; при равенстве — более позднюю) выбирает код, а куратор пишет к ней название и этикетку —
зная всю историю попыток, он может сделать её частью сюжета.

In [ ]:
def show_image(path, caption=""):
    display(HTML(f'<figure style="margin:8px 0"><img src="{path}" width="420">'
                 f'<figcaption style="font-size:90%;color:#666">{caption}</figcaption></figure>'))


def create_work(work):
    LOG.clear()
    for a in (artist, engineer):
        a.reset()   # для нового слова художник и инженер начинают диалог с чистого листа

    iterations, prev = [], None
    for it in range(MAX_ITERATIONS):
        display(Markdown(f"### «{work['word']}» — версия {it + 1}"))
        prompt = discuss(work, it, prev)
        prompt, path = render(work, it, prompt)
        show_image(path, prompt)
        seen, verdict = evaluate(work, it, path)
        prev = {"n": it + 1, "prompt": prompt, "image": path.relative_to(OUT_DIR).as_posix(),
                "critic": seen, "judge": verdict}
        iterations.append(prev)
        if verdict["score"] >= TARGET_SCORE:
            break

    best = max(range(len(iterations)), key=lambda i: (iterations[i]["judge"]["score"], i))
    history = "\n".join(f"Версия {x['n']}: {x['judge']['score']}/10, зритель увидел: {x['critic']['feeling']}"
                        for x in iterations)
    label = ask_json(curator,
        f"Слово: «{work['word']}» ({work['lang']}), смысл: {work['meaning']}\n"
        f"Попытки нейросетей:\n{history}\n"
        f"На выставку выбрана версия {best + 1}. Её промпт: {iterations[best]['prompt']}\n"
        f"Описание картины глазами зрителя: {iterations[best]['critic']['description']}\n\n"
        'Ответь JSON: {"title": "название картины (не само слово)", '
        '"label": "этикетка, 3-4 предложения: что изображено, как это связано со словом, что показали попытки"}')
    say(curator.name, f"**«{label.get('title', '')}»**\n\n{label.get('label', '')}", best)

    return {**work, "iterations": iterations, "best": best,
            "title": label.get("title", work["word"]), "label": label.get("label", ""),
            "dialogue": list(LOG)}

## 7. Коллекция

Пять слов из пяти языков — от почти «визуального» (*komorebi* — это буквально свет сквозь листву) до
предельно абстрактного (*тоска*, *hiraeth*). Так видно, где многоагентной системе легко, а где обратная связь
действительно нужна.

In [ ]:
WORDS = [
    {"slug": "toska", "word": "Тоска", "lang": "русский",
     "meaning": "Щемящая душевная боль без определённой причины; смутное беспокойство, томление, тяга к чему-то "
                "неясному и недостижимому, скука и ностальгия одновременно. Набоков писал, что ни одно английское "
                "слово не передаёт всех её оттенков."},
    {"slug": "komorebi", "word": "Komorebi (木漏れ日)", "lang": "японский",
     "meaning": "Солнечный свет, пробивающийся сквозь листву деревьев, — пятна света и тени, и то тихое ощущение "
                "покоя и мимолётной красоты, которое он вызывает."},
    {"slug": "hiraeth", "word": "Hiraeth", "lang": "валлийский",
     "meaning": "Тоска по дому, в который невозможно вернуться или которого, возможно, никогда не было; "
                "ностальгия по утраченному месту, времени или себе прежнему, смешанная с горем."},
    {"slug": "mono-no-aware", "word": "Mono no aware (物の哀れ)", "lang": "японский",
     "meaning": "Тихая, светлая печаль от осознания быстротечности всего сущего; умение ценить красоту вещей "
                "именно потому, что они преходящи (как цветение сакуры, которое длится несколько дней)."},
    {"slug": "sobremesa", "word": "Sobremesa", "lang": "испанский",
     "meaning": "Время после окончания трапезы, когда никто не спешит вставать из-за стола: люди продолжают "
                "разговаривать, смеяться, пить кофе — радость неспешного общения и близости."},
]

## 8. Запуск

Результаты сохраняются **после каждого слова** в `docs/data/gallery.json` и `docs/data/gallery.js` — второй файл
нужен, чтобы сайт открывался и локально, без веб-сервера. Уже готовые слова при повторном запуске пропускаются
(чтобы перерисовать слово, удалите его из `gallery.json` или поставьте `REDO = True`).

In [ ]:
REDO = False
GALLERY_JSON = DATA_DIR / "gallery.json"

def save_gallery(gallery):
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    text = json.dumps(gallery, ensure_ascii=False, indent=1)
    GALLERY_JSON.write_text(text, encoding="utf-8")
    (DATA_DIR / "gallery.js").write_text(f"window.GALLERY = {text};\n", encoding="utf-8")

gallery = json.loads(GALLERY_JSON.read_text(encoding="utf-8")) if GALLERY_JSON.exists() and not REDO else {"works": []}
gallery.update({"title": "Непереводимое", "models": MODELS, "image_model": IMAGE_MODEL,
                "max_iterations": MAX_ITERATIONS, "target_score": TARGET_SCORE})
done = {w["slug"] for w in gallery["works"]}

for word in WORDS:
    if word["slug"] in done:
        print(f"«{word['word']}» уже готово — пропускаем")
        continue
    display(Markdown(f"---\n## {word['word']} ({word['lang']})"))
    started = time.time()
    result = create_work(word)
    result["seconds"] = round(time.time() - started)
    gallery["works"] = [w for w in gallery["works"] if w["slug"] != word["slug"]] + [result]
    order = [x["slug"] for x in WORDS]
    gallery["works"].sort(key=lambda w: order.index(w["slug"]) if w["slug"] in order else len(order))
    gallery["generated_at"] = datetime.datetime.now().isoformat(timespec="seconds")
    save_gallery(gallery)

## 9. Предисловие к выставке

Куратор смотрит на всю коллекцию целиком (названия, баллы, историю попыток) и пишет вступительный текст для сайта.

In [ ]:
summary = "\n".join(
    f"- «{w['word']}»: картина «{w['title']}», баллы по версиям: "
    + " → ".join(str(x["judge"]["score"]) for x in w["iterations"])
    for w in gallery["works"])
intro = curator.ask(
    "Вот работы выставки, где нейросети рисовали непереводимые слова, а слепой зритель-нейросеть угадывал их смысл:\n"
    f"{summary}\n\nНапиши вступительный текст к выставке: 1 абзац, 4-5 предложений. Без заголовка.", remember=False)
gallery["intro"] = intro
save_gallery(gallery)
display(Markdown(intro))

## 10. Итоги: помогает ли обратная связь?

Таблица показывает балл судьи для каждой версии. Если петля обратной связи работает, баллы должны расти от версии к версии.

In [ ]:
rows = ["| Слово | Название | Баллы по версиям | Лучшая |", "|---|---|---|---|"]
for w in gallery["works"]:
    scores = " → ".join(str(x["judge"]["score"]) for x in w["iterations"])
    rows.append(f"| {w['word']} | {w['title']} | {scores} | v{w['best'] + 1} |")
display(Markdown("\n".join(rows)))

first = [w["iterations"][0]["judge"]["score"] for w in gallery["works"]]
best = [w["iterations"][w["best"]]["judge"]["score"] for w in gallery["works"]]
if first:
    print(f"Средний балл первой версии: {sum(first) / len(first):.1f}, лучшей версии: {sum(best) / len(best):.1f}")

In [ ]:
# Галерея прямо в ноутбуке: все версии каждого слова, рамкой выделена выбранная куратором
html = []
for w in gallery["works"]:
    html.append(f"<h3>{w['word']} — «{w['title']}»</h3><p style='max-width:900px'>{w['label']}</p><div style='display:flex;gap:8px'>")
    for i, x in enumerate(w["iterations"]):
        border = "4px solid #c9a227" if i == w["best"] else "4px solid transparent"
        html.append(f"<figure style='margin:0;width:260px'><img src='{OUT_DIR}/{x['image']}' width='260' style='border:{border}'>"
                    f"<figcaption style='font-size:80%'>v{x['n']}: <b>{x['judge']['score']}/10</b> — "
                    f"зритель: {x['critic'].get('feeling', '')}</figcaption></figure>")
    html.append("</div>")
display(HTML("".join(html)))

## Сайт

Все результаты лежат в папке `docs/`, которая публикуется через GitHub Pages (Settings → Pages → Branch `main`, папка `/docs`).
Сайт (`docs/index.html`) читает `docs/data/gallery.js` и показывает для каждого слова лучшую картину, этикетку куратора,
динамику баллов, все версии и полный диалог агентов. Локально его можно открыть просто двойным щелчком по `docs/index.html`.

Выводы по работе — в [REPORT.md](REPORT.md).